# ✅ **HISEMOTIONS at IberLEF 2026: Historical Text-Based Emotion Detection in Early Modern Spanish Correspondence**


# 1️⃣ Ollama


In [ ]:
import requests

try:
    response = requests.get("http://localhost:11434/api/tags")
    if response.status_code == 200:
        print("Ollama is running and reachable from Jupyter!")
        print("Installed models:")
        for model in response.json().get("models", []):
            print("-", model["name"])
    else:
        print("Ollama responded, but not OK:", response.status_code)
except Exception as e:
    print("Could not connect to Ollama:", e)


# 2️⃣ Define input/output file paths

In [ ]:
input_path = r"PostScriptim.csv" 
output_path = r"output.csv"

# 3️⃣ Load your CSV

In [ ]:
import pandas as pd
df = pd.read_csv(input_path)

if "text" not in df.columns:
    raise ValueError("The CSV must contain a column named 'text'.")

texts = df["text"].tolist()

# 4️⃣ Define the prompt

In [ ]:
prompt_template = """
Task: Emotion classification of 16th-century Spanish letters.

You must classify the emotions expressed in the following fragment of a 16th-century letter written in Spanish.

This is a multi-label classification task using EXACTLY the following seven categories:
anger, disgust, fear, joy, sadness, surprise, hope

ANNOTATION GUIDELINES:

1. Annotate ONLY emotions expressed by the letter's author.
   - Ignore emotions attributed to third parties mentioned in the text.

2. Prioritize emotions expressed at the moment of writing.
   - Do NOT annotate emotions referring only to past events reported narratively unless the emotion is clearly re-experienced in the present writing moment.

3. Do NOT annotate non-emotional discourse.
   - Courtesy formulas, greetings, openings, closings, blessings, and other structural epistolary conventions must be assigned 0 in all categories.

4. If no emotional content is present according to these rules, return all zeros (baseline case).

BASELINE EXAMPLE (no emotion expressed):

Text:
"Muy magnífico señor, beso las manos de vuestra merced y quedo como su servidor."

Output:
0,0,0,0,0,0,0

ANNOTATED EXAMPLE:

Text:
"Me hallo con gran tristeza por vuestra ausencia, mas confío en Dios que presto tornaremos a vernos."

Interpretation:
- sadness (present emotional state)
- hope (confidence about the future)

Output:
0,0,0,1,0,1

OUTPUT FORMAT REQUIREMENTS:

- Return ONLY seven binary values.
- Use 1 if the emotion is present, 0 if not.
- Follow EXACTLY this order:
anger, disgust, fear, joy, sadness, surprise, hope
- Separate values with commas.
- Do NOT include explanations, reasoning, labels, or extra text.
- Do NOT add spaces.
- Do NOT add line breaks.
- Output must contain exactly 7 comma-separated digits.

Text: {}
"""


# 5️⃣ Run inference using Ollama

In [ ]:
from ollama import chat
from ollama import ChatResponse



In [ ]:
results = []

for i, text in enumerate(texts, start=1):
    print(f"Processing {i}/{len(texts)}...")

    prompt = prompt_template.format(text)


    response: ChatResponse = chat(
        model="gemma3:latest",  
        messages=[{"role": "user", "content": prompt}]
    )

    # Access output
    output_text = response.message.content.strip()
    results.append(output_text)


# 6️⃣ Save results



In [ ]:
df["emotions"] = results
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\n Classification complete!")
print(f"Results saved to: {output_path}")